In [ ]:
import signal
import pandas as pd
import metadisassembler5 as medi5
import pickle, geass, molecular_structure_clustering

from rdkit import Chem
from rdkit.Chem import rdChemReactions
from rdkit.Chem.Draw import rdMolDraw2D
from progress_bar import print_progress_bar

In [ ]:
'''
# Loads the PBU
with open('metadisassembler5/data/PBU_v2.pkl', 'rb') as f:
    PBU = pickle.load(f)
'''

In [ ]:
'''
# Loads the BUL
with open('metadisassembler5/data/BUL_Mar23.pkl', 'rb') as f:
    BULv2 = pickle.load(f)
'''

In [ ]:
df_database = pd.read_csv('SECRETed_Database_Dec23.csv')
database_mols = dict(zip(df_database['ID_unique'], df_database['Isomeric_SMILES']))

In [ ]:
def create_medi_library(lib_mols):
    '''
    Function that creates a Metabolic Disassembler molecule Library
    
    Inputs:
    - lib_mols: Molecule dictionary (format: {mol_id: mol (SMILES or rdkit mol)})
    
    Outputs:
    - Result Metabolic Disassembler Library
    '''
    result_lib = medi5.Library.Library()
    for i, (name, mol) in enumerate(lib_mols.items()):
        print(f'Building Library: [{i}/{len(lib_mols)}]', end='\r')
        if type(mol) == str:
            cpd = medi5.Compound.Compound()
            cpd.input_rdkmol(Chem.MolFromSmiles(mol))
            result_lib._append_cpd(cpd, name)
        elif type(mol) == Chem.rdchem.Mol:
            cpd = medi5.Compound.Compound()
            cpd.input_rdkmol(mol)
            result_lib._append_cpd(cpd, name)
        elif type(mol) == list:
            for m in mol:
                if type(m) == str:
                    cpd = medi5.Compound.Compound()
                    cpd.input_rdkmol(Chem.MolFromSmiles(m))
                    result_lib._append_cpd(cpd, name)
                elif type(m) == Chem.rdchem.Mol:
                    cpd = medi5.Compound.Compound()
                    cpd.input_rdkmol(m)
                    result_lib._append_cpd(cpd, name)
    return result_lib
            

In [ ]:
fragmentation_scheme = {
       'CH3' : '[CH3;X4]',
       'CH2' : '[CH2;X4]',
       'CH' : '[CH1;X4]',
       'C' : ['[CH0;X4]' , '[CH0;X3]'],
       'CH2=CH' : '[CH2]=[CH]',
       'CH=CH' : '[CH]=[CH]',
       'CH2=C' : ['[CH2]=[C]' , '[CH2]=[c]'],
       'CH=C' : ['[CH]=[CH0]' , '[CH]=[cH0]'],
       'ACH' : '[cH]',
       'AC' : '[cH0]',
       'ACCH3' : '[c][CH3;X4]',
       'ACCH2' : '[c][CH2;X4]',
       'ACCH' : '[c][CH;X4]',
       'OH' : '[OH]',
       'CH3OH' : '[CH3][OH]',
       'H2O' : '[OH2]',
       'ACOH' : '[c][OH]',
       'CH3CO' : '[CH3][CH0]=O',
       'CH2CO' : '[CH2][CH0]=O',
       'CH=O' : '[CH]=O',
       'CH3COO' : '[CH3]C=O[OH0]',
       'CH2COO' : '[CH2]C=O[OH0]',
       'HCOO' : '[CH]=O[OH0]',
       'CH3O' : '[CH3][OH0]',
       'CH2O' : '[CH2][OH0]',
       'CHO' : '[CH][OH0]',
       'THF' : '[CH2;R][OH0]',
       'CH3NH2' : '[CH3][NH2]',
       'CH2NH2' : '[CH2][NH2]',
       'CHNH2' : '[CH][NH2]',
       'CH3NH' : '[CH3][NH]',
       'CH2NH' : '[CH2][NH]',
       'CHNH' : '[CH][NH]',
       'CH3N' :  ['[CH3][N]' , '[CH3][n]'],
       'CH2N' : '[CH2][N]',
       'ACNH2' : '[c][NH2]',
       'C5H5N' : 'n1[cH][cH][cH][cH][cH]1',
       'C5H4N' : ['n1[c][cH][cH][cH][cH]1',
              'n1[cH][c][cH][cH][cH]1',
              'n1[cH][cH][c][cH][cH]1'],
       'C5H3N' : ['n1[c][c][cH][cH][cH]1',
              'n1[c][cH][c][cH][cH]1',
              'n1[c][cH][cH][c][cH]1',
              'n1[c][cH][cH][cH][c]1',
              'n1[cH][c][c][cH][cH]1',
              'n1[cH][c][cH][c][cH]1'],
       'CH3CN' : '[CH3]C#N',
       'CH2CN' : '[CH2]C#N',
       'COOH' : 'C=O[OH]',
       'HCOOH' : '[CH]=O[OH]',
       'CH2Cl' : '[CH2]Cl',
       'CHCl' : '[CH]Cl',
       'CCl' : '[CH0]Cl',
       'CH2Cl2' : '[CH2]ClCl',
       'CHCl2' : '[CH]ClCl',
       'CCl2' : 'CClCl',
       'CHCl3' : '[CH]ClClCl',
       'CCl3' : 'CClClCl',
       'CCl4' : 'CClClClCl',
       'ACCl' : '[c]Cl',
       'CH3NO2' : '[CH3][N+]=O[O-]',
       'CH2NO2' : '[CH2][N+]=O[O-]',
       'CHNO2' : '[CH][N+]=O[O-]',
       'ACNO2' : '[c][N+]=O[O-]',
       'CS2' : 'C=S=S',
       'CH3SH' : '[CH3][SH]',
       'CH2SH' : '[CH2][SH]',
       'Furfural' : 'O=[CH]c1[cH][cH][cH]o1',
       'DOH' : '[OH][CH2][CH2][OH]',
       'I' : '[IH0]',
       'Br' : '[BrH0]',
       'CH#C' : '[CH]#C',
       'C#C' : 'C#C',
       'DMSO' : '[CH3]S=O[CH3]',
       'ACRY' : '[CH2]=[CH1][C]#N',
       'C=C' : '[CH0]=[CH0]',
       'ACF' : '[c]F',
       'DMF' : '[CH]=ON[CH3][CH3]',
       'CF3' : 'CFFF',
       'CF2' : 'CFF',
       'CF' : '[C]F',
       'COO' : ['[CH0]=O[OH0]', '[cH0]=O[oH0]'],
       'SiH3' : '[SiH3]',
       'SiH2' : '[SiH2]',
       'SiH' : '[SiH]',
       'Si' : '[Si]',
       'SiH2O' : '[SiH2][OH0]',
       'SiHO' : '[SiH][OH0]',
       'SiO' : '[Si][OH0]',
       'CCl3F' : 'CClClClF',
       'CCl2F' : 'CClClF',
       'HCCl2F' : '[CH]ClClF',
       'HCClF' : '[CH]ClF',
       'CClF2' : 'CClFF',
       'HCClF2' : '[CH]ClFF',
       'CClF3' : 'CClFFF',
       'CCl2F2' : 'CClClFF',
       'CONH2' : 'C=O[NH2]',
       'CONHCH3' : 'C=O[NH][CH3]',
       'CONHCH2' : 'C=O[NH][CH2]',
       'CONCH32' : 'C=ON[CH3][CH3]',
       'CONCH3CH2' : 'C=ON[CH3][CH2]',
       'CONCH22' : 'C=ON[CH2][CH2]',
       'CH3S' : '[CH3]S',
       'CH2S' : '[CH2]S',
       'CHS' : '[CH]S',
       'MORPH' : '[CH2]1[CH2][NH][CH2][CH2]O1',
       'C4H4S' : '[cH]1[cH][s;X2][cH][cH]1',
       'C4H3S' : ['[c]1[cH][s;X2][cH][cH]1',
              '[cH]1[c][s;X2][cH][cH]1'],
       'C4H2S' : ['[c]1[c][s;X2][cH][cH]1',
              '[c]1[cH][s;X2][cH][c]1',
              '[cH]1[c][s;X2][c][cH]1',
              '[cH]1[c][s;X2][cH][c]1'],
       'NCO' : 'N=C=O',
       'CH22SU' : '[CH2]S=O=O[CH2]',
       'CH2CHSU' : '[CH2]S=O=O[CH]'
}

In [ ]:
'''
fails = []
indexer = {}
results = {}
fragments = {}
fragment_names = {}
n_samples = len(mols)

# -------------------------- Generate the query_specific_bul (qsu) -----------------------------------

# We first compute the query_specific_bul. We save it for later computations. The results obtained 
# from this configuration are not expected to provide good results, only to provide an initialization
# step that saves computational times.

temp_qsu={} # The qsu are saved in this auxiliar memory.


for j, molecule_smiles in not_NP_fragments.items():
    # Iterate over the molecules 
    
    clear_output()
    print(f'PROGRESS: [{j+1}/{len(not_NP_fragments)}]')
    print(f'SMILES: {molecule_smiles}')
    
    disassembler = medi5.MetaDisassembler5() # Init the disassembler
    disassembler.output_folder = "./metadisassembler_v3_results" # Choose the output folder
    disassembler.useChirality = False # Whether to use Chilarity. IT HAS NOT BEEN TESTED IN 'TRUE' MODE
    disassembler.pbu = PBU # Assign the pbu
    disassembler.bul = BULv2 # Assign the bul
    disassembler.precomputed_qsu = False # We are not computed the qsu yet.
    disassembler.n_img = 1 
    disassembler.time_limit = 1
    name = str(j) + '_' + indexer_original_mols[idx]  # The name of the molecule
    molecule = Chem.MolFromSmiles(molecule_smiles) # The molecule from the current SMILES
    disassembler.input_molecule(molecule, name) # Define the input with the molecular structure and the name
     
    try:
        disassembler.disassemble() # disassemble
        indexer[j] = disassembler.name
        results[j] = disassembler.result
        temp_qsu[j] = disassembler.query_specific_bul # IMPORTANT: SAVE THE QSU
        fragment_names[j] = [disassembler.query_specific_bul.names[disassembler.match_bu_idx[k]] for aux, k in enumerate(disassembler.result[0]["target"])]
        fragments[j] = [disassembler.query_specific_bul.cpds[disassembler.match_bu_idx[k]] for aux, k in enumerate(disassembler.result[0]["target"])]
    except:
        print('Error')
        indexer[j] = 'Error'
        results[j] = 'Error'
        temp_qsu[j] = 'Error'
        fragment_names[j] = 'Error'

    
    if len(disassembler.result) == 0:
        fails.append(True)
    else:
        fails.append(False)
'''

In [ ]:
def execute_metabolic_disassembler(mols, PBU, BUL, output_folder="./metadisassembler_v3_results"):
    '''
    Function that executes the Metabolic Disassembler using a given BUL and PBU for a set of molecules.
    
    Inputs:
    - mols: Dictionary of molecule smiles (format: {mol_id: mol_SMILES})
    - PBU: Metabolic Disassembler Library containing the PBU molecules
    - BUL: Metabolic Disassembler Library containing the BUL molecules
    - output_folder: Folder where the Metabolic Disassembler results will be saved to
    '''
    fragments = {}
    fragment_names = {}
    results = {}
    temp_qsu = {}

    print_progress_bar(0, len(mols), suffix=f'(0/{len(mols)})')
    # Run metabolic disassembler
    for j, (mol_id, molecule_smiles) in enumerate(mols.items()):
        # Iterate over the molecules 

        disassembler = medi5.MetaDisassembler5() # Init the disassembler
        disassembler.output_folder = output_folder # Choose the output folder
        disassembler.useChirality = False # Whether to use Chilarity. IT HAS NOT BEEN TESTED IN 'TRUE' MODE
        disassembler.pbu = PBU # Assign the pbu
        disassembler.bul = BUL # Assign the bul
        disassembler.precomputed_qsu = False # We are not computed the qsu yet.
        disassembler.n_img = 1 
        disassembler.time_limit = 1
        name = mol_id  # The name of the molecule
        molecule = Chem.MolFromSmiles(molecule_smiles) # The molecule from the current SMILES
        disassembler.input_molecule(molecule, name) # Define the input with the molecular structure and the name

        try:
            # Define the timeout for disassembling
            signal.alarm(60)
            disassembler.disassemble() # disassemble
            # Cancel the timer if the function finished before timeout
            signal.alarm(0)
            results[j] = disassembler.result
            temp_qsu[j] = disassembler.query_specific_bul # IMPORTANT: SAVE THE QSU
            fragment_names[j] = [disassembler.query_specific_bul.names[disassembler.match_bu_idx[k]] for aux, k in enumerate(disassembler.result[0]["target"])]
            fragments[j] = [disassembler.query_specific_bul.cpds[disassembler.match_bu_idx[k]] for aux, k in enumerate(disassembler.result[0]["target"])]
        except:
            print('Error')
            indexer[j] = 'Error'
            results[j] = 'Error'
            temp_qsu[j] = 'Error'
            fragments[j] = 'Error'
            fragment_names[j] = 'Error'
        print_progress_bar(j + 1, len(mols), suffix=f'({j}/{len(mols)})')
    return fragments, fragment_names, results, temp_qsu

In [ ]:
'''
disassembler = medi5.MetaDisassembler5() # Init the disassembler
disassembler.output_folder = output_folder # Choose the output folder
disassembler.useChirality = False # Whether to use Chilarity. IT HAS NOT BEEN TESTED IN 'TRUE' MODE
disassembler.pbu = PBU # Assign the pbu
disassembler.bul = BUL # Assign the bul
disassembler.precomputed_qsu = False # We are not computed the qsu yet.
disassembler.n_img = 1 
disassembler.time_limit = 1
name = str(j) + '_' + indexer_original_mols[mol_idx]  # The name of the molecule
molecule = Chem.MolFromSmiles(molecule_smiles) # The molecule from the current SMILES
disassembler.input_molecule(molecule, name) # Define the input with the molecular structure and the name

try:
    # Define the timeout for disassembling
    signal.alarm(60)
    disassembler.disassemble() # disassemble
    # Cancel the timer if the function finished before timeout
    signal.alarm(0)
    fragment_names[j] = [disassembler.query_specific_bul.names[disassembler.match_bu_idx[k]] for aux, k in enumerate(disassembler.result[0]["target"])]
    fragments[j] = [disassembler.query_specific_bul.cpds[disassembler.match_bu_idx[k]] for aux, k in enumerate(disassembler.result[0]["target"])]
except:
    print('Error')
    fragments[j] = 'Error'
    fragment_names[j] = 'Error'
'''

In [ ]:
PBU = create_medi_library(fragmentation_scheme)
BUL = create_medi_library(fragmentation_scheme)
fragments, fragment_names, results, temp_qsu = execute_metabolic_disassembler(database_mols, PBU, BUL)